# 07 – Build Trainer Datasets (Prompt + Image → Model Label)

This notebook:

1. Loads the final router datasets (`router_{split}_final.parquet`) built with the chosen
   utility scheme and hierarchical weights.
2. Joins them with the image table (`cauldron_images.parquet`) containing in-memory PNG bytes.
3. Builds minimal trainer datasets for each split (`train`, `validation`, `test`) with:
   - `input_text` (prompt_raw)
   - `image_png` (PNG bytes, **no disk images**)
   - `label_id` / `label_name` (model to route to)
4. Adds sanity checks and simple plots so we can see what’s happening.

> **Important:** This notebook assumes that:
> - `05_final_data_prepare.ipynb` has already been run,
> - and that it produced `router_train_final.parquet`, `router_validation_final.parquet`,
>   and `router_test_final.parquet` under `FINAL_DATA_DIR`.
> - `04_prep_img_dataset.ipynb` has been run and produced `cauldron_images.parquet`
>   with a column `image_png` that stores PNG bytes.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from dataclasses import dataclass
import matplotlib.pyplot as plt
from typing import Dict, Tuple

# For image preview from bytes
from PIL import Image
import io

plt.rcParams["figure.figsize"] = (7, 4)


@dataclass
class PerfWeightsHier:
    w_sample: float
    w_task: float
    w_global: float


In [ ]:
# ------------------------------------------------------------------
# PATH CONFIG (EDIT THESE TO MATCH YOUR ENV)
# ------------------------------------------------------------------
# Base directory where your Parquet files live
# TODO: update this path to your actual base directory
DATA_ROOT = Path.cwd().parent.parent.parent / "dataset"
# print(Path.cwd().parents[:1])
print(f" DATA_ROOT = {DATA_ROOT}")

# Final router datasets from 05_final_data_prepare.ipynb
FINAL_DATA_DIR = DATA_ROOT / "final_dataset" / "router_final"           # e.g. contains router_train_final.parquet

# Image table from 04_prep_img_dataset.ipynb (with PNG bytes)
IMAGE_TABLE_PATH = DATA_ROOT /  "images" / "cauldron_images.parquet"  # <-- CHANGE ME if different

print(f"Images table path: {IMAGE_TABLE_PATH}")

In [ ]:
# Output directory for trainer datasets
TRAINER_OUT_DIR =  DATA_ROOT / "final_dataset" /  "router_lexico"
TRAINER_OUT_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = ["train", "val", "test"]


In [ ]:

# ------------------------------------------------------------------
# Utility + hierarchical weights (for reproducibility metadata)
# ------------------------------------------------------------------
UTILITY_SCHEME = "linear"
HIER_WEIGHTS = PerfWeightsHier(
    w_sample=0.6,   # Sample-level performance dominates
    w_task=0.35,    # Task-level priors help with task-specific patterns
    w_global=0.05,  # Global priors provide weak baseline
)

print("DATA_ROOT        :", DATA_ROOT)
print("FINAL_DATA_DIR   :", FINAL_DATA_DIR)
print("IMAGE_TABLE_PATH :", IMAGE_TABLE_PATH)
print("TRAINER_OUT_DIR  :", TRAINER_OUT_DIR)
print()
print("UTILITY_SCHEME   :", UTILITY_SCHEME)
print("HIER_WEIGHTS     :", HIER_WEIGHTS)

In [ ]:
# Load image table with PNG bytes
img_table = pd.read_parquet(IMAGE_TABLE_PATH)

print("Image table loaded.")
print("Shape:", img_table.shape)
print("Columns:", list(img_table.columns))

print("\nFirst 3 rows of image table:")
display(img_table.head(3))

In [ ]:


# Sanity check expected columns
required_img_cols = {"image_bytes_hash", "source_config"}
missing = required_img_cols - set(img_table.columns)
if missing:
    raise ValueError(f"Image table is missing required columns: {missing}")

# For this pipeline we assume PNG bytes column is called 'image_png'
if "image_png" not in img_table.columns:
    raise ValueError(
        "Expected an 'image_png' column with PNG bytes. "
        "Update this notebook if your column name differs."
    )

# Keep only columns we really need for join + training
img_table = img_table[["image_bytes_hash", "source_config", "image_png"]]
print("\nTrimmed image table columns:", list(img_table.columns))

In [ ]:
def load_router_final(split: str) -> pd.DataFrame:
    """
    Load router_{split}_final.parquet produced by 05_final_data_prepare.ipynb
    and run basic sanity checks.
    """
    path = FINAL_DATA_DIR / f"router_{split}_final.parquet"
    print(f"\n=== Loading router final dataset for split = '{split}' ===")
    print("Path:", path)

    df = pd.read_parquet(path)
    print("Shape:", df.shape)
    print("Columns:", list(df.columns)[:20], " ...")  # show first few

    # Check required columns
    required_cols = {
        "sample_id",
        "prompt_raw",
        "image_bytes_hash",
        "source_config",
        "router_best_model_id",
        "router_best_model_name",
    }
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Router final dataset for split '{split}' is missing columns: {missing}")

    # Quick peek
    display(df.head(3))

    return df



In [ ]:
def build_trainer_split(
    split: str,
    df_router: pd.DataFrame,
    img_table: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Join router final dataset with image table, then build trainer-ready
    dataset for the given split.

    df_router is expected to already contain:
      - core input cols (sample_id, prompt_raw, image_bytes_hash, etc.)
      - metadata cols (router_task, source_dataset, etc.)
      - label cols (router_best_model_id, router_best_model_name, etc.)

    Returns:
      df_joined : df_router with image bytes attached
      df_trainer: final trainer parquet view
    """
    print(f"\n=== Building trainer dataset for split = '{split}' ===")

    n_before = len(df_router)
    print(f"Rows in router_{split}_final:", n_before)

    # -----------------------------
    # 1. Join with image table
    # -----------------------------
    join_keys = ["image_bytes_hash", "source_config"]

    # Ensure each image appears only once in the image table
    img_dedup = img_table.drop_duplicates(subset=join_keys, keep="first")
    print(f"Image table: {len(img_table)} rows -> {len(img_dedup)} unique image rows")

    df_joined = df_router.merge(
        img_dedup,
        on=join_keys,
        how="inner",
        # Many router rows can map to one image; each (hash, config) is unique in img_dedup
        validate="many_to_one",
    )

    n_after = len(df_joined)
    print(f"Rows after join with image table: {n_after}")

    if n_after < n_before:
        print(
            f"WARNING: lost {n_before - n_after} rows during join. "
            "These samples have no matching image entry."
        )

    # -----------------------------
    # 2. Define column groups
    #    (match your Step 8 config)
    # -----------------------------
    # Core router input features (things we may actually feed the model)
    core_input_cols = [
        "sample_id",
        "image_path",
        "image_bytes_hash",          # for merge / image lookup
        "prompt_raw",
        "img_width",
        "img_height",
        "img_aspect_ratio",
        "txt_prompt_length_chars",
        "txt_prompt_length_words",
    ]
    core_input_cols = [c for c in core_input_cols if c in df_joined.columns]

    # Metadata / analysis-only cols (kept for eval, not necessarily fed to router)
    metadata_cols = [
        "router_task",
        "source_dataset",
        "source_config",
        "txt_question_type",
        "txt_has_mc_options",
        "ground_truth",
        "ground_truth_type",
    ]
    metadata_cols = [c for c in metadata_cols if c in df_joined.columns]

    # Label columns
    label_cols = [
        "router_best_model_id",
        "router_best_model_name",
        "router_chosen_perf",
        "router_chosen_cost",
    ]
    label_cols = [c for c in label_cols if c in df_joined.columns]

    # -----------------------------
    # 3. Build trainer dataframe
    # -----------------------------
    # Start with all the cols we explicitly care about
    cols_keep = core_input_cols + metadata_cols + label_cols

    base = {c: df_joined[c] for c in cols_keep}

    # Image bytes (what the model will actually use)
    base["image_png"] = df_joined["image_png"]

    # For training, we’ll use prompt_raw as input_text and best_model_* as label
    # but we keep the original columns too for analysis.
    if "prompt_raw" in df_joined.columns:
        base["input_text"] = df_joined["prompt_raw"]
    else:
        # Fallback: if prompt_raw was renamed in your pipeline
        raise ValueError("Expected 'prompt_raw' in df_joined for input_text.")

    if "router_best_model_id" in df_joined.columns:
        base["label_id"] = df_joined["router_best_model_id"].astype("int32")
    if "router_best_model_name" in df_joined.columns:
        base["label_name"] = df_joined["router_best_model_name"].astype("string")

    df_trainer = pd.DataFrame(base)

    # Utility metadata (same for all rows)
    df_trainer["utility_scheme"] = UTILITY_SCHEME
    df_trainer["hier_w_sample"] = HIER_WEIGHTS.w_sample
    df_trainer["hier_w_task"] = HIER_WEIGHTS.w_task
    df_trainer["hier_w_global"] = HIER_WEIGHTS.w_global

    # Optional: copy soft label columns if present (router_soft_p_*)
    soft_cols = [c for c in df_joined.columns if c.startswith("router_soft_p_")]
    if soft_cols:
        print(f"Found {len(soft_cols)} soft-label columns. Adding them to trainer dataset.")
        df_trainer = pd.concat([df_trainer, df_joined[soft_cols]], axis=1)
    else:
        print("No soft-label columns (router_soft_p_*) found.")

    # -----------------------------
    # 4. Save + preview
    # -----------------------------
    out_path = TRAINER_OUT_DIR / f"router_{split}_trainer.parquet"
    df_trainer.to_parquet(out_path, index=False)
    print(f"Saved trainer dataset for split '{split}' to:\n  {out_path}")
    print("Trainer dataset shape:", df_trainer.shape)

    display(df_trainer.head(5))

    return df_joined, df_trainer


In [ ]:
router_final_by_split: Dict[str, pd.DataFrame] = {}
joined_by_split: Dict[str, pd.DataFrame] = {}
trainer_by_split: Dict[str, pd.DataFrame] = {}

for split in SPLITS:
    df_router = load_router_final(split)
    router_final_by_split[split] = df_router

    df_joined, df_trainer = build_trainer_split(split, df_router, img_table)
    joined_by_split[split] = df_joined
    trainer_by_split[split] = df_trainer

print("\n=== Done building trainer datasets for all splits. ===")
for split in SPLITS:
    print(
        f"{split:>10}: "
        f"{len(router_final_by_split[split])} router rows -> "
        f"{len(trainer_by_split[split])} trainer rows"
    )

In [ ]:
print("=== Checking sample_id overlaps across splits ===")

id_sets = {split: set(df["sample_id"]) for split, df in trainer_by_split.items()}

for a in SPLITS:
    for b in SPLITS:
        if a >= b:
            continue
        overlap = id_sets[a] & id_sets[b]
        print(f"Overlap between {a} and {b}: {len(overlap)}")
        if overlap:
            print("  WARNING: there should be no overlap in sample_id across splits.")

In [ ]:
print("=== Label distribution per split (label_id) ===")

for split, df_trainer in trainer_by_split.items():
    print(f"\n--- {split.upper()} ---")
    counts = df_trainer["label_id"].value_counts().sort_index()
    print(counts)

    fig, ax = plt.subplots()
    counts.plot(kind="bar", ax=ax)
    ax.set_title(f"Label distribution for {split} (label_id)")
    ax.set_xlabel("label_id (model index)")
    ax.set_ylabel("count")
    plt.tight_layout()
    plt.show()

    # If label_name is present, also show mapping
    label_map = (
        df_trainer[["label_id", "label_name"]]
        .drop_duplicates()
        .sort_values("label_id")
    )
    print("\nLabel id → name mapping:")
    display(label_map)

In [ ]:
print("=== Task distribution per split (router_task, if available) ===")

for split, df_trainer in trainer_by_split.items():
    print(f"\n--- {split.upper()} ---")
    if "router_task" not in df_trainer.columns:
        print("router_task column not found, skipping.")
        continue

    counts = df_trainer["router_task"].value_counts()
    print(counts)

    fig, ax = plt.subplots()
    counts.plot(kind="bar", ax=ax)
    ax.set_title(f"Task distribution for {split}")
    ax.set_xlabel("router_task")
    ax.set_ylabel("count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
print("=== Visual sanity check: decode 1 random sample per split ===")

def show_random_sample(df_trainer: pd.DataFrame, split: str, seed: int = 0):
    # Filter to rows that have non-null image_png
    mask = df_trainer["image_png"].notna()
    if not mask.any():
        print(f"[{split}] No rows with image_png. Skipping preview.")
        return

    row = df_trainer[mask].sample(1, random_state=seed).iloc[0]

    img_bytes = row["image_png"]
    try:
        img = Image.open(io.BytesIO(img_bytes))
    except Exception as e:
        print(f"[{split}] Failed to decode image bytes:", e)
        return

    print(f"\n[{split.upper()}] sample_id: {row['sample_id']}")
    print(f"label_id: {row['label_id']}, label_name: {row['label_name']}")
    print("input_text (truncated):")
    text = row["input_text"]
    print(text[:300], "..." if len(text) > 300 else "")

    if "router_task" in df_trainer.columns:
        print("router_task:", row["router_task"])

    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"{split} preview")
    plt.show()


for i, split in enumerate(SPLITS):
    show_random_sample(trainer_by_split[split], split, seed=i)

## Summary

- Built trainer datasets:
  - `router_train_trainer.parquet`
  - `router_validation_trainer.parquet`
  - `router_test_trainer.parquet`
- Each row contains:
  - `sample_id`
  - `input_text` (prompt_raw)
  - `image_png` (PNG bytes, ready to decode in the training pipeline)
  - `label_id` / `label_name` (selected model)
  - `utility_scheme`, `hier_w_sample`, `hier_w_task`, `hier_w_global`
  - Optional soft label columns (`router_soft_p_*`) if present
- Sanity:
  - No (or very few) overlaps across splits
  - Reasonable label and task distributions
  - Image bytes correctly decode for random samples